In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/latest.pt
/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt
/kaggle/input/datasets/achiver07/data-project/sc_val.jsonl
/kaggle/input/datasets/achiver07/data-project/acts_val.jsonl
/kaggle/input/datasets/achiver07/data-project/acts_train.jsonl
/kaggle/input/datasets/achiver07/data-project/tribunals_val.jsonl
/kaggle/input/datasets/achiver07/data-project/test.json
/kaggle/input/datasets/achiver07/data-project/sc_train.jsonl
/kaggle/input/datasets/achiver07/data-project/tribunals_train.jsonl
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-cle

In [2]:
# ================================================================
# ARCHITECTURE 4 — FAST INFERENCE + DECODING TUNER
#
# What this does
#   1. Batched generation WITH A KV CACHE (local layers = true sliding window
#      of 256, global layers = full cache). ~10-50x faster than re-running the
#      whole context for every token, so many decoding strategies are testable.
#   2. Decoding options: greedy / temperature+top-k+top-p / repetition penalty
#      (generated tokens only) / no-repeat n-gram / MBR (sample N candidates,
#      keep the one that agrees most with the others, using ROUGE-L).
#   3. TUNE: scores each strategy on held-out validation passages with local
#      ROUGE-L and BLEU-4 (pure Python, no pretrained model) so you can pick
#      the best BEFORE spending a submission.
#   4. TEST: runs the best (or chosen) strategy on test.json and writes
#      predictions with exactly TARGET_WORDS whitespace words each.
#
# Pure PyTorch + numpy. No pretrained weights. Your own tokenizer.json is
# loaded with the `tokenizers` engine if installed (it only reads your file).
# ================================================================

import os, sys, json, math, time, re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------
CKPT_PATH = ("/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/"
             "default/1/arch4_hybrid_gqa_201m/best.pt")   # auto-searched if this path is wrong
DATA_DIR = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1"
OUT_DIR = "/kaggle/working/inference_outputs"             # /kaggle/input is read-only
TEST_JSON_CANDIDATES = [                                   # first one with usable text prompts wins
    os.path.join(DATA_DIR, "test.json"),
    "/kaggle/input/datasets/achiver07/data-project/test.json",
]

TARGET_WORDS = 128            # whitespace words per prediction (your evaluator showed 128)
MAX_NEW_TOKENS = 300          # hard cap; generation stops early once every row has enough words
MAX_BATCH = 64                # total decode rows per batch (prompts x samples); lower if OOM
ADD_SPECIAL_TOKENS = False    # False: encode prompts as raw text (no BOS/EOS added by the tokenizer)
BOS_ID = None                 # set an int if your training data started documents with a BOS token
BAN_IDS = None                # None = auto-ban EOS-like tokens so we always reach TARGET_WORDS

RUN_SELFTEST = True           # checks cached decoding == full forward (recommended)
RUN_TUNE = False             # already done: mbr8 won. Set True to re-compare configs
RUN_TEST = True

TUNE_EXAMPLES = 200
TUNE_PROMPT_TOKENS = "auto"   # "auto" = median test-prompt length if measurable, else 256
TUNE_SELECT_BY = "rougeL"     # "rougeL" or "bleu4"
SOURCE_NAMES = ["acts", "sc", "tribunals"]
SOURCE_PROBS = [0.273, 0.469, 0.258]

DEFAULT_DEC = {"temperature": 0.0, "top_k": 0, "top_p": 1.0,
               "rep_pen": 1.0, "no_repeat": 0, "n": 1}
CONFIGS = [
    {"name": "sample_t0.8_p0.95", "temperature": 0.8, "top_k": 50, "top_p": 0.95},
    {"name": "greedy"},
    {"name": "greedy+norep4", "rep_pen": 1.05, "no_repeat": 4},
    {"name": "t0.3+norep4", "temperature": 0.3, "top_p": 0.9, "rep_pen": 1.05, "no_repeat": 4},
    {"name": "mbr8_t0.7", "temperature": 0.7, "top_p": 0.95, "rep_pen": 1.05, "no_repeat": 4, "n": 8},
    {"name": "mbr16_t0.7", "temperature": 0.7, "top_p": 0.95, "rep_pen": 1.05, "no_repeat": 4, "n": 16},
]
FINAL_CONFIG = "mbr8_t0.7"            # a config name, or "auto" = best from TUNE
FALLBACK_CONFIG = "greedy+norep4"     # used when "auto" but TUNE was skipped
SEED = 1234

# ----------------------------------------------------------------
# DEVICE
# ----------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
CACHE_DTYPE = torch.float16 if USE_AMP else torch.float32
print("Device:", DEVICE)


def autocast():
    return torch.amp.autocast(DEVICE.type, enabled=USE_AMP, dtype=torch.float16)


# ================================================================
# ---- METRICS BEGIN (pure Python; approximations of the evaluator) ----
# ================================================================
_rouge_re = re.compile(r"[a-z0-9]+")
_bleu_re = re.compile(r"\w+|[^\w\s]")


def norm_tokens(text):
    """rouge-score style: lowercase, alphanumeric tokens only."""
    return _rouge_re.findall(text.lower())


def lcs_len(a, b):
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b, 1):
            if x == y:
                cur.append(prev[j - 1] + 1)
            else:
                cur.append(prev[j] if prev[j] > cur[j - 1] else cur[j - 1])
        prev = cur
    return prev[-1]


def rouge_l_f1(a, b):
    if not a or not b:
        return 0.0
    l = lcs_len(a, b)
    if l == 0:
        return 0.0
    p, r = l / len(a), l / len(b)
    return 2 * p * r / (p + r)


def _ngrams(toks, n):
    return Counter(tuple(toks[i:i + n]) for i in range(len(toks) - n + 1))


def corpus_bleu4(preds, refs):
    """Corpus BLEU-4 on regex word/punct tokens, single reference, crude smoothing."""
    match, total, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        p, r = _bleu_re.findall(p), _bleu_re.findall(r)
        pl += len(p); rl += len(r)
        for n in range(1, 5):
            pc, rc = _ngrams(p, n), _ngrams(r, n)
            match[n - 1] += sum(min(c, rc[g]) for g, c in pc.items())
            total[n - 1] += max(len(p) - n + 1, 0)
    if min(total) == 0 or pl == 0:
        return 0.0
    prec = [(m if m > 0 else 1e-9) / t for m, t in zip(match, total)]
    bp = 1.0 if pl > rl else math.exp(1 - rl / pl)
    return bp * math.exp(sum(math.log(x) for x in prec) / 4)


def mbr_pick(texts):
    """Minimum-Bayes-risk: index of the candidate most similar (ROUGE-L) to all others."""
    toks = [norm_tokens(t) for t in texts]
    n = len(toks)
    score = [0.0] * n
    for i in range(n):
        for j in range(i + 1, n):
            f = rouge_l_f1(toks[i], toks[j])
            score[i] += f
            score[j] += f
    return max(range(n), key=lambda i: score[i])


class NGramBan:
    """Per-sequence no-repeat-n-gram table over GENERATED tokens only."""

    def __init__(self, n_seqs, n):
        self.k = n - 1
        self.hist = [[] for _ in range(n_seqs)]
        self.tab = [dict() for _ in range(n_seqs)]

    def banned(self):
        rows, cols = [], []
        for b, (h, t) in enumerate(zip(self.hist, self.tab)):
            if len(h) >= self.k:
                s = t.get(tuple(h[-self.k:]))
                if s:
                    rows.extend([b] * len(s))
                    cols.extend(s)
        return rows, cols

    def update(self, toks):
        for b, tk in enumerate(toks):
            h = self.hist[b]
            if len(h) >= self.k:
                self.tab[b].setdefault(tuple(h[-self.k:]), set()).add(tk)
            h.append(tk)
# ================================================================
# ---- METRICS END ----
# ================================================================


# ----------------------------------------------------------------
# MODEL  (identical to training; same state_dict keys)
# ----------------------------------------------------------------
def rotate_half(x):
    h = x.shape[-1] // 2
    return torch.cat([-x[..., h:], x[..., :h]], dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base):
        super().__init__()
        inv = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        f = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv)
        emb = torch.cat([f, f], dim=-1)
        self.register_buffer("cos", emb.cos(), persistent=False)
        self.register_buffer("sin", emb.sin(), persistent=False)

    def forward(self, T):
        return self.cos[:T][None, None], self.sin[:T][None, None]


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.in_proj = nn.Linear(d_model, 2 * d_ff, bias=False)
        self.out_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        g, u = self.in_proj(x).chunk(2, dim=-1)
        return self.out_proj(F.silu(g) * u)


class GQAAttention(nn.Module):
    def __init__(self, d_model, n_q, n_kv, window, is_global):
        super().__init__()
        self.n_q, self.n_kv = n_q, n_kv
        self.head_dim = d_model // n_q
        self.rep = n_q // n_kv
        self.window, self.is_global = window, is_global
        q_dim, kv_dim = n_q * self.head_dim, n_kv * self.head_dim
        self.split = [q_dim, kv_dim, kv_dim]
        self.qkv_proj = nn.Linear(d_model, q_dim + 2 * kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, mask):          # full-sequence path (training layout)
        B, T, C = x.shape
        Hq, Hkv, Dh, rep = self.n_q, self.n_kv, self.head_dim, self.rep
        q, k, v = torch.split(self.qkv_proj(x), self.split, dim=-1)
        q = q.view(B, T, Hq, Dh).transpose(1, 2)
        k = k.view(B, T, Hkv, Dh).transpose(1, 2)
        v = v.view(B, T, Hkv, Dh).transpose(1, 2)
        cos, sin = cos.to(q.dtype), sin.to(q.dtype)
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        if self.is_global:
            k = k[:, :, None].expand(B, Hkv, rep, T, Dh).reshape(B, Hq, T, Dh)
            v = v[:, :, None].expand(B, Hkv, rep, T, Dh).reshape(B, Hq, T, Dh)
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            return self.out_proj(y.transpose(1, 2).reshape(B, T, C))
        W = self.window
        N = T // W
        qc = q.view(B, Hq, N, W, Dh).transpose(1, 2).reshape(B * N, Hq, W, Dh)

        def chunk_kv(t):
            tc = t.view(B, Hkv, N, W, Dh)
            prev = torch.cat([torch.zeros_like(tc[:, :, :1]), tc[:, :, :-1]], dim=2)
            t2 = torch.cat([prev, tc], dim=3).transpose(1, 2)[:, :, :, None]
            return t2.expand(B, N, Hkv, rep, 2 * W, Dh).reshape(B * N, Hq, 2 * W, Dh)

        y = F.scaled_dot_product_attention(qc, chunk_kv(k), chunk_kv(v), attn_mask=mask)
        y = y.view(B, N, Hq, W, Dh).permute(0, 1, 3, 2, 4).reshape(B, T, C)
        return self.out_proj(y)


class Arch4Block(nn.Module):
    def __init__(self, cfg, idx):
        super().__init__()
        self.is_global = (idx + 1) % cfg["GLOBAL_EVERY"] == 0
        self.norm1 = nn.LayerNorm(cfg["D_MODEL"])
        self.attn = GQAAttention(cfg["D_MODEL"], cfg["N_Q_HEADS"], cfg["N_KV_HEADS"],
                                 cfg["WINDOW"], self.is_global)
        self.norm2 = nn.LayerNorm(cfg["D_MODEL"])
        self.ffn = SwiGLU(cfg["D_MODEL"], cfg["D_FF"])

    def forward(self, x, cos, sin, mask):
        x = x + self.attn(self.norm1(x), cos, sin, mask)
        return x + self.ffn(self.norm2(x))


class Architecture4(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        W = cfg["WINDOW"]
        self.window, self.max_seq_len = W, cfg["MAX_SEQ_LEN"]
        self.token_embedding = nn.Embedding(cfg["VOCAB_SIZE"], cfg["D_MODEL"])
        self.layers = nn.ModuleList([Arch4Block(cfg, i) for i in range(cfg["N_LAYERS"])])
        self.final_norm = nn.LayerNorm(cfg["D_MODEL"])
        self.lm_head = nn.Linear(cfg["D_MODEL"], cfg["VOCAB_SIZE"], bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.rope = RotaryEmbedding(cfg["D_MODEL"] // cfg["N_Q_HEADS"], cfg["MAX_SEQ_LEN"], cfg["ROPE_BASE"])
        n_chunks = cfg["MAX_SEQ_LEN"] // W
        i = torch.arange(W)[:, None]
        j = torch.arange(2 * W)[None, :]
        m = (j > i) & (j <= i + W)
        mask = m[None].repeat(n_chunks, 1, 1)
        mask[0] &= (j >= W)
        self.register_buffer("chunk_mask", mask[:, None], persistent=False)

    def hidden_states(self, input_ids):            # reference full forward (T % W == 0)
        B, T = input_ids.shape
        W = self.window
        assert T % W == 0 and T <= self.max_seq_len
        N = T // W
        x = self.token_embedding(input_ids)
        cos, sin = self.rope(T)
        mask = self.chunk_mask[:N][None].expand(B, N, 1, W, 2 * W).reshape(B * N, 1, W, 2 * W)
        for b in self.layers:
            x = b(x, cos, sin, mask)
        return self.final_norm(x)


DEFAULT_CFG = {"VOCAB_SIZE": 40000, "D_MODEL": 1024, "N_LAYERS": 12, "N_Q_HEADS": 16,
               "N_KV_HEADS": 8, "D_FF": 3328, "MAX_SEQ_LEN": 1024, "SEQ_LEN": 1024,
               "WINDOW": 256, "GLOBAL_EVERY": 3, "ROPE_BASE": 10000.0}


def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception as e:
        print("weights_only load failed (", str(e)[:80], ") -> retrying without it (your own file).")
        return torch.load(path, map_location="cpu", weights_only=False)


def find_checkpoint(path):
    if os.path.exists(path):
        return path
    import glob
    hits = []
    for pat in ("/kaggle/input/**/best.pt", "/kaggle/working/**/best.pt"):
        hits += glob.glob(pat, recursive=True)
    if not hits:
        raise FileNotFoundError(
            f"No checkpoint at {path} and no best.pt found under /kaggle/input or /kaggle/working. "
            f"Add your trained model to this notebook (Add Input -> Models) and re-run.")
    print(f"Checkpoint not at the configured path; using {hits[0]}")
    if len(hits) > 1:
        print("  other best.pt files found:", hits[1:])
    return hits[0]


CKPT_PATH = find_checkpoint(CKPT_PATH)
print(f"\nLoading {CKPT_PATH} ...")
_ck = torch_load(CKPT_PATH)
cfg = {**DEFAULT_CFG, **_ck.get("cfg", {})}
model = Architecture4(cfg)
model.load_state_dict(_ck["model"])
model.to(DEVICE).eval()
print(f"Loaded: step {_ck.get('step')} | stored val {_ck.get('best_val')}")
del _ck
VOCAB, MAXLEN, W = cfg["VOCAB_SIZE"], cfg["MAX_SEQ_LEN"], cfg["WINDOW"]
os.makedirs(OUT_DIR, exist_ok=True)


# ----------------------------------------------------------------
# TOKENIZER (your own tokenizer.json)
# ----------------------------------------------------------------
class Tok:
    def __init__(self, path):
        self.kind, self.tk, self.id2tok, self.byte_level = "none", None, {}, False
        if not os.path.exists(path):
            print("Tokenizer file not found:", path)
            return
        try:
            from tokenizers import Tokenizer
            self.tk = Tokenizer.from_file(path)
            self.kind = "hf"
            return
        except Exception as e:
            print(f"`tokenizers` unavailable/unreadable ({str(e)[:70]}); using decode-only fallback.")
        try:
            j = json.load(open(path, "r", encoding="utf-8"))
            vocab = j["model"]["vocab"]
            if isinstance(vocab, dict):
                self.id2tok = {int(i): t for t, i in vocab.items()}
            else:
                self.id2tok = {i: (p[0] if isinstance(p, list) else p) for i, p in enumerate(vocab)}
            for a in j.get("added_tokens", []):
                self.id2tok[int(a["id"])] = a["content"]
            self.byte_level = "ByteLevel" in json.dumps([j.get("decoder"), j.get("pre_tokenizer")])
            if self.byte_level:
                bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
                cs, n = bs[:], 0
                for b in range(256):
                    if b not in bs:
                        bs.append(b); cs.append(256 + n); n += 1
                self.u2b = {chr(c): b for b, c in zip(bs, cs)}
            self.kind = "decode_only"
        except Exception as e:
            print("Could not parse tokenizer.json:", str(e)[:100])

    def can_encode(self):
        return self.kind == "hf"

    def can_decode(self):
        return self.kind in ("hf", "decode_only")

    def encode(self, text):
        return self.tk.encode(text, add_special_tokens=ADD_SPECIAL_TOKENS).ids

    def decode(self, ids):
        ids = [int(i) for i in ids]
        if self.kind == "hf":
            return self.tk.decode(ids, skip_special_tokens=False)
        s = "".join(self.id2tok.get(i, "") for i in ids)
        if self.byte_level:
            return bytes(self.u2b[c] for c in s if c in self.u2b).decode("utf-8", errors="replace")
        return s.replace("\u2581", " ")

    def find_eos(self):
        names = ["<|endoftext|>", "</s>", "<eos>", "[EOS]", "<|eos|>", "<EOS>"]
        for n in names:
            if self.kind == "hf":
                i = self.tk.token_to_id(n)
                if i is not None:
                    return i
            elif self.kind == "decode_only":
                inv = {t: i for i, t in self.id2tok.items()}
                if n in inv:
                    return inv[n]
        return None


tok = Tok(os.path.join(DATA_DIR, "tokenizer.json"))
print("Tokenizer mode:", tok.kind)
if not tok.can_decode():
    sys.exit("This script needs to decode tokens to text (to count words / score). "
             "Make sure tokenizer.json is readable (install `tokenizers` if needed).")
if BAN_IDS is None:
    _e = tok.find_eos()
    BAN_IDS = [_e] if _e is not None else []
print("Banned token ids:", BAN_IDS)


def words_of(ids):
    return tok.decode(ids).split()


def words_text(ids):
    return " ".join(words_of(ids)[:TARGET_WORDS])


# ================================================================
# FAST CACHED GENERATOR
# ================================================================
class State:
    def __init__(self, kc, vc, pad_len, P, valid):
        self.kc, self.vc, self.pad_len, self.P, self.valid = kc, vc, pad_len, P, valid


class FastGen:
    def __init__(self, m, cfg):
        self.m = m
        a0 = m.layers[0].attn
        self.Hq, self.Hkv, self.Dh, self.rep = a0.n_q, a0.n_kv, a0.head_dim, a0.rep
        self.is_global = [b.is_global for b in m.layers]
        self.cos, self.sin = m.rope.cos, m.rope.sin
        self.slots = torch.arange(MAXLEN, device=DEVICE)

    def _rope(self, pos):
        return self.cos[pos][:, None], self.sin[pos][:, None]      # (B,1,T,Dh)

    def _layer(self, i, x, c, s, kc, vc, start, mask):
        blk = self.m.layers[i]
        a = blk.attn
        B, T, C = x.shape
        Hq, Hkv, Dh, rep = self.Hq, self.Hkv, self.Dh, self.rep
        h = blk.norm1(x)
        q, k, v = torch.split(a.qkv_proj(h), a.split, dim=-1)
        q = q.view(B, T, Hq, Dh).transpose(1, 2)
        k = k.view(B, T, Hkv, Dh).transpose(1, 2)
        v = v.view(B, T, Hkv, Dh).transpose(1, 2)
        c, s = c.to(q.dtype), s.to(q.dtype)
        q = q * c + rotate_half(q) * s
        k = k * c + rotate_half(k) * s
        kc[:, :, start:start + T] = k
        vc[:, :, start:start + T] = v
        L = start + T
        K, V = kc[:, :, :L], vc[:, :, :L]
        if T == 1:                                   # decode: fold the GQA group into the query axis
            qf = q.reshape(B, Hkv, rep, Dh)
            y = F.scaled_dot_product_attention(qf, K, V, attn_mask=mask)
            y = y.reshape(B, Hq, 1, Dh)
        else:                                        # prefill
            Ke = K[:, :, None].expand(B, Hkv, rep, L, Dh).reshape(B, Hq, L, Dh)
            Ve = V[:, :, None].expand(B, Hkv, rep, L, Dh).reshape(B, Hq, L, Dh)
            y = F.scaled_dot_product_attention(q, Ke, Ve, attn_mask=mask)
        y = y.transpose(1, 2).reshape(B, T, C)
        x = x + a.out_proj(y)
        return x + blk.ffn(blk.norm2(x))

    @torch.no_grad()
    def prefill(self, prompts, max_new):
        B = len(prompts)
        P = max(len(p) for p in prompts)
        Tmax = P + max_new
        assert Tmax <= MAXLEN, f"prompt({P}) + max_new({max_new}) exceeds context {MAXLEN}"
        ids = torch.zeros(B, P, dtype=torch.long, device=DEVICE)
        pad_len = torch.tensor([P - len(p) for p in prompts], device=DEVICE)
        for b, p in enumerate(prompts):
            ids[b, P - len(p):] = torch.tensor(p, device=DEVICE)
        idx = torch.arange(P, device=DEVICE)
        pos = (idx[None, :] - pad_len[:, None]).clamp(min=0)
        c, s = self._rope(pos)
        valid_pre = idx[None, :] >= pad_len[:, None]
        causal = idx[None, :] <= idx[:, None]
        window = idx[None, :] > idx[:, None] - W
        base = causal[None, None] & valid_pre[:, None, None, :]
        eye = torch.eye(P, dtype=torch.bool, device=DEVICE)[None, None]   # pad rows stay finite
        mg = base | eye
        ml = (base & window[None, None]) | eye
        kc = [torch.zeros(B, self.Hkv, Tmax, self.Dh, dtype=CACHE_DTYPE, device=DEVICE) for _ in self.m.layers]
        vc = [torch.zeros_like(k) for k in kc]
        with autocast():
            x = self.m.token_embedding(ids)
            for i in range(len(self.m.layers)):
                x = self._layer(i, x, c, s, kc[i], vc[i], 0, mg if self.is_global[i] else ml)
            logits = F.linear(self.m.final_norm(x[:, -1]), self.m.lm_head.weight).float()
        valid = torch.arange(Tmax, device=DEVICE)[None, :] >= pad_len[:, None]
        return State(kc, vc, pad_len, P, valid), logits

    @torch.no_grad()
    def expand(self, st, n, logits):
        st.kc = [k.repeat_interleave(n, 0) for k in st.kc]
        st.vc = [v.repeat_interleave(n, 0) for v in st.vc]
        st.pad_len = st.pad_len.repeat_interleave(n, 0)
        st.valid = st.valid.repeat_interleave(n, 0)
        return logits.repeat_interleave(n, 0)

    @torch.no_grad()
    def step(self, st, tok_ids, slot):
        pos = (slot - st.pad_len)[:, None]
        c, s = self._rope(pos)
        vm = st.valid[:, :slot + 1]
        mg = vm[:, None, None, :]
        ml = (vm & (self.slots[:slot + 1] > slot - W)[None, :])[:, None, None, :]
        with autocast():
            x = self.m.token_embedding(tok_ids[:, None])
            for i in range(len(self.m.layers)):
                x = self._layer(i, x, c, s, st.kc[i], st.vc[i], slot, mg if self.is_global[i] else ml)
            return F.linear(self.m.final_norm(x[:, 0]), self.m.lm_head.weight).float()


def sample_next(lg, dec, gen):
    if dec["temperature"] <= 0:
        return lg.argmax(-1)
    lg = lg / dec["temperature"]
    if dec["top_k"] and dec["top_k"] > 0:
        kth = torch.topk(lg, min(dec["top_k"], lg.size(-1))).values[:, -1:]
        lg = lg.masked_fill(lg < kth, float("-inf"))
    if dec["top_p"] < 1.0:
        sl, si = torch.sort(lg, descending=True, dim=-1)
        pr = F.softmax(sl, dim=-1)
        remove = (pr.cumsum(-1) - pr) > dec["top_p"]
        sl = sl.masked_fill(remove, float("-inf"))
        lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
    return torch.multinomial(F.softmax(lg, dim=-1), 1, generator=gen).squeeze(1)


gen_engine = FastGen(model, cfg)
rng_torch = torch.Generator(device=DEVICE).manual_seed(SEED)


@torch.no_grad()
def generate_batch(prompts, n, dec, max_new):
    st, logits = gen_engine.prefill(prompts, max_new)
    if n > 1:
        logits = gen_engine.expand(st, n, logits)
    B = logits.size(0)
    out = torch.zeros(B, max_new, dtype=torch.long, device=DEVICE)
    rp = dec["rep_pen"]
    seen = torch.zeros(B, VOCAB, dtype=torch.bool, device=DEVICE) if rp != 1.0 else None
    ng = NGramBan(B, dec["no_repeat"]) if dec["no_repeat"] >= 2 else None
    ar = torch.arange(B, device=DEVICE)
    t_end = max_new
    for t in range(max_new):
        lg = logits
        if seen is not None:                                    # penalise GENERATED tokens only
            pen = torch.where(lg > 0, lg / rp, lg * rp)
            lg = torch.where(seen, pen, lg)
        if BAN_IDS:
            lg[:, BAN_IDS] = float("-inf")
        if ng is not None:
            rows, cols = ng.banned()
            if rows:
                lg[torch.tensor(rows, device=DEVICE), torch.tensor(cols, device=DEVICE)] = float("-inf")
        nxt = sample_next(lg, dec, rng_torch)
        out[:, t] = nxt
        if seen is not None:
            seen[ar, nxt] = True
        if ng is not None:
            ng.update(nxt.tolist())
        if (t + 1) % 32 == 0 and t + 1 < max_new:               # stop once every row has enough words
            cur = out[:, :t + 1].tolist()
            if min(len(words_of(r)) for r in cur) >= TARGET_WORDS + 3:
                t_end = t + 1
                break
        if t == max_new - 1:
            break
        logits = gen_engine.step(st, nxt, st.P + t)
    return out[:, :t_end].tolist()


def run_decoding(prompts, dec, label=""):
    n = dec["n"] if dec["temperature"] > 0 else 1
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    per_batch = max(1, MAX_BATCH // n)
    preds = [None] * len(prompts)
    t0 = time.time()
    for bi in range(0, len(order), per_batch):
        idxs = order[bi:bi + per_batch]
        outs = generate_batch([prompts[i] for i in idxs], n, dec, MAX_NEW_TOKENS)
        for k, i in enumerate(idxs):
            cands = [words_text(outs[k * n + j]) for j in range(n)]
            preds[i] = cands[0] if n == 1 else cands[mbr_pick(cands)]
        done = min(bi + per_batch, len(order))
        el = time.time() - t0
        print(f"   [{label}] {done}/{len(order)} prompts | {el:.0f}s | "
              f"ETA {el / done * (len(order) - done):.0f}s", flush=True)
    return preds


# ----------------------------------------------------------------
# SELF-TEST: cached decoding must match the full forward pass
# ----------------------------------------------------------------
def load_val():
    arrs = {}
    for nme in SOURCE_NAMES:
        p = os.path.join(DATA_DIR, f"{nme}_val.bin" if nme != "tribunals" else "tribunals_val_f.bin")
        assert os.path.exists(p), f"Missing {p}"
        arrs[nme] = np.memmap(p, dtype=np.uint16, mode="r")
    return arrs


val_arrays = load_val()


@torch.no_grad()
def full_logits(seq):
    n = len(seq)
    T = ((n + W - 1) // W) * W
    inp = torch.zeros(1, T, dtype=torch.long, device=DEVICE)
    inp[0, :n] = torch.tensor(seq, device=DEVICE)
    with autocast():
        h = model.hidden_states(inp)
        return F.linear(h[0, n - 1], model.lm_head.weight).float()


@torch.no_grad()
def self_test():
    print("\n" + "=" * 70 + "\nSELF-TEST (KV-cache decode vs full forward)\n" + "=" * 70)
    d = val_arrays["sc"]
    A = d[1000:1300].astype(np.int64).tolist()         # 300 tokens: window + padding paths exercised
    Bq = d[50000:50180].astype(np.int64).tolist()      # 180 tokens: needs left padding vs A
    refs = [full_logits(A), full_logits(Bq)]
    st, _ = gen_engine.prefill([A[:-1], Bq[:-1]], 4)
    lg = gen_engine.step(st, torch.tensor([A[-1], Bq[-1]], device=DEVICE), st.P)
    ok = True
    for name, ref, got in zip(["A(300 tok)", "B(180 tok)"], refs, lg):
        diff = (ref - got).abs().max().item()
        same = int(ref.argmax()) == int(got.argmax())
        in_top3 = int(ref.argmax()) in got.topk(3).indices.tolist()   # tolerate fp16 near-ties
        print(f"  {name}: max|logit diff| = {diff:.4f} | same top-1: {same} | ref top-1 in cached top-3: {in_top3}")
        ok = ok and diff < 1.0 and in_top3
    print("  RESULT:", "PASS" if ok else "FAIL (do not trust cached generation; tell me this output)")
    return ok


# ----------------------------------------------------------------
# TEST SET LOADING
# ----------------------------------------------------------------
PROMPT_KEYS = ["prompt", "input", "text", "context", "source", "document", "question", "query"]
ID_KEYS = ["id", "idx", "example_id", "uid", "index"]


def _is_token_list(v):
    return isinstance(v, list) and len(v) > 0 and all(isinstance(x, int) for x in v)


def load_test(path):
    """Handles: list of strings / list of dicts / {id: example} dicts / {"data": ...} wrappers /
    jsonl. Prompts may be text or already-tokenized id lists."""
    raw = open(path, "r", encoding="utf-8").read()
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        data = [json.loads(l) for l in raw.splitlines() if l.strip()]
    if isinstance(data, dict):                      # unwrap {"data": [...]}-style containers
        for k in ("data", "examples", "test", "instances", "items", "records"):
            if isinstance(data.get(k), (list, dict)) and len(data[k]) > 1:
                data = data[k]
                break
    if isinstance(data, dict):                      # {id: example} mapping
        items, shape = list(data.items()), "dict {id: example}"
    else:
        items, shape = list(enumerate(data)), "list [example, ...]"
    ids, prompts, pkey = [], [], None
    for key, ex in items:
        rid = key
        if isinstance(ex, dict):
            if pkey is None:
                pkey = next((k for k in PROMPT_KEYS if isinstance(ex.get(k), str)), None)
                if pkey is None:
                    pkey = next((k for k, v in ex.items() if _is_token_list(v)), None)
                if pkey is None:
                    strs = [(len(v), k) for k, v in ex.items() if isinstance(v, str)]
                    if not strs:
                        raise ValueError(f"no text/token field in example with keys {list(ex.keys())}")
                    pkey = max(strs)[1]
            ikey = next((k for k in ID_KEYS if k in ex), None)
            if ikey:
                rid = ex[ikey]
            val = ex[pkey]
        else:
            val = ex
        if not (isinstance(val, str) or _is_token_list(val)):
            raise ValueError(f"unsupported prompt value ({type(val).__name__}) for id {key!r}")
        ids.append(rid)
        prompts.append(val)
    if not prompts:
        raise ValueError("no examples found")
    first = prompts[0]
    kind = "token ids" if _is_token_list(first) else "text"
    preview = first[:200] if isinstance(first, str) else str(first[:20]) + " ..."
    print(f"test.json: {len(prompts)} examples | structure: {shape} | prompt field: {pkey!r} | prompts are {kind}")
    print(f"  first id: {ids[0]!r}\n  first prompt: {preview!r}")
    if isinstance(items[0][1], dict):
        print("  example keys:", list(items[0][1].keys()))
    return ids, prompts


def encode_prompts(texts):
    max_prompt = MAXLEN - MAX_NEW_TOKENS
    out, cut = [], 0
    for t in texts:
        if isinstance(t, str):
            ids = tok.encode(t)
            if BOS_ID is not None:
                ids = [BOS_ID] + ids
        else:                                       # already token ids
            ids = [int(x) for x in t]
        if len(ids) > max_prompt:
            ids, cut = ids[-max_prompt:], cut + 1
        out.append(ids if ids else [0])
    lens = np.array([len(x) for x in out])
    print(f"  prompt tokens: min/median/max = {lens.min()}/{int(np.median(lens))}/{lens.max()} "
          f"| truncated (kept the END of the prompt): {cut}")
    return out


# ----------------------------------------------------------------
# TUNE
# ----------------------------------------------------------------
def dec_of(c):
    return {**DEFAULT_DEC, **c}


def run_tune(test_prompt_lens=None):
    print("\n" + "=" * 70 + "\nTUNE (local proxy: ROUGE-L + BLEU-4 on held-out validation)\n" + "=" * 70)
    if TUNE_PROMPT_TOKENS == "auto":
        p_tokens = int(np.median(test_prompt_lens)) if test_prompt_lens else 256
    else:
        p_tokens = int(TUNE_PROMPT_TOKENS)
    p_tokens = max(16, min(p_tokens, MAXLEN - MAX_NEW_TOKENS))
    ref_tokens = 320
    print(f"  {TUNE_EXAMPLES} validation passages | prompt {p_tokens} tokens | reference = next {TARGET_WORDS} words")
    rng = np.random.RandomState(777)
    probs = np.array(SOURCE_PROBS) / np.sum(SOURCE_PROBS)
    prompts, refs = [], []
    for _ in range(TUNE_EXAMPLES):
        nme = SOURCE_NAMES[rng.choice(len(SOURCE_NAMES), p=probs)]
        d = val_arrays[nme]
        s = int(rng.randint(0, len(d) - p_tokens - ref_tokens - 1))
        seg = d[s:s + p_tokens + ref_tokens].astype(np.int64).tolist()
        prompts.append(seg[:p_tokens])
        refs.append(" ".join(words_of(seg[p_tokens:])[:TARGET_WORDS]))
    rows = []
    for c in CONFIGS:
        dec = dec_of(c)
        t0 = time.time()
        preds = run_decoding(prompts, dec, c["name"])
        per = [rouge_l_f1(norm_tokens(p), norm_tokens(r)) for p, r in zip(preds, refs)]
        rl = float(np.mean(per))
        bl = corpus_bleu4(preds, refs)
        rows.append({"name": c["name"], "rougeL": rl, "bleu4": bl, "sec": time.time() - t0,
                     "cfg": c, "per": per})
        print(f"  -> {c['name']:22s} ROUGE-L {rl:.4f} | BLEU-4 {bl:.4f} | {time.time() - t0:.0f}s")
        torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
    print("\n  Config                 ROUGE-L   BLEU-4    time")
    for r in sorted(rows, key=lambda r: -r[TUNE_SELECT_BY]):
        print(f"  {r['name']:22s} {r['rougeL']:.4f}   {r['bleu4']:.4f}   {r['sec']:.0f}s")
    best = max(rows, key=lambda r: r[TUNE_SELECT_BY])
    print(f"\n  BEST by {TUNE_SELECT_BY}: {best['name']}")
    print(f"  ROUGE-L gap of each config to the best, paired over the same {TUNE_EXAMPLES} passages "
          f"(mean +/- SE; a gap well above ~2 SE is a real difference):")
    for r in rows:
        if r is best:
            continue
        d = np.array(best["per"]) - np.array(r["per"])
        print(f"    best - {r['name']:22s} {d.mean():+.4f} +/- {d.std(ddof=1) / math.sqrt(len(d)):.4f}")
    print("  (local proxy only: absolute numbers differ from the official scorer; "
          "BERTScore is not computed because it needs a pretrained model.)")
    json.dump([{k: v for k, v in r.items() if k != "per"} for r in rows],
              open(os.path.join(OUT_DIR, "tune_results.json"), "w"), indent=2)
    return best["name"]


# ----------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------
test_ids = test_prompt_ids = None
test_texts = None
if (RUN_TEST or (RUN_TUNE and TUNE_PROMPT_TOKENS == "auto")) and tok.can_encode():
    print("\nReading test prompts ...")
    for _p in TEST_JSON_CANDIDATES:
        if not os.path.exists(_p):
            continue
        try:
            test_ids, test_texts = load_test(_p)
            if test_texts:
                print("  using", _p)
                break
        except Exception as e:
            print(f"  could not use {_p}: {str(e)[:100]}")
    if test_texts:
        test_prompt_ids = encode_prompts(test_texts)
if RUN_TEST and not test_prompt_ids:
    print("\nWARNING: cannot run TEST (no usable text prompts in test.json, or the tokenizer "
          "cannot encode; `tokenizers` is needed to encode prompts).")
    RUN_TEST = False

if RUN_SELFTEST and not self_test():
    sys.exit("Self-test failed; stopping.")

chosen = FINAL_CONFIG
if RUN_TUNE:
    best_name = run_tune([len(x) for x in test_prompt_ids] if test_prompt_ids else None)
    if FINAL_CONFIG == "auto":
        chosen = best_name
elif FINAL_CONFIG == "auto":
    chosen = FALLBACK_CONFIG

if RUN_TEST:
    cmap = {c["name"]: c for c in CONFIGS}
    assert chosen in cmap, f"Unknown config {chosen!r}; options: {list(cmap)}"
    print("\n" + "=" * 70 + f"\nTEST PREDICTIONS with config: {chosen}\n" + "=" * 70)
    preds = run_decoding(test_prompt_ids, dec_of(cmap[chosen]), "test")
    counts = [len(p.split()) for p in preds]
    print(f"  words per prediction: min/avg/max = {min(counts)}/{np.mean(counts):.1f}/{max(counts)} "
          f"| shorter than {TARGET_WORDS}: {sum(c < TARGET_WORDS for c in counts)}")
    json.dump([{"id": i, "prediction": p} for i, p in zip(test_ids, preds)],
              open(os.path.join(OUT_DIR, "predictions_records.json"), "w"), ensure_ascii=False, indent=1)
    json.dump(preds, open(os.path.join(OUT_DIR, "predictions_list.json"), "w"), ensure_ascii=False, indent=1)
    with open(os.path.join(OUT_DIR, "predictions.jsonl"), "w", encoding="utf-8") as f:
        for i, p in zip(test_ids, preds):
            f.write(json.dumps({"id": i, "prediction": p}, ensure_ascii=False) + "\n")
    print(f"  Saved predictions_records.json / predictions_list.json / predictions.jsonl in {OUT_DIR}")
    _first = test_texts[0] if isinstance(test_texts[0], str) else tok.decode(test_texts[0])
    print("\n  Example 1\n  PROMPT END:", _first[-200:].replace("\n", " "),
          "\n  PREDICTION:", preds[0][:300])

Device: cuda

Loading /kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt ...
Loaded: step 10000 | stored val 2.475979009747505
Tokenizer mode: hf
Banned token ids: [2]

Reading test prompts ...
test.json: 1000 examples | structure: dict {id: example} | prompt field: 'text' | prompts are text
  first id: '1'
  first prompt: 'The issue involved in these petitions is as to whether the acquisition proceedings can be challenged at a belated stage. The issue is no more res integra as the issue has been considered by this Court'
  example keys: ['text']
  using /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
  prompt tokens: min/median/max = 540/602/724 | truncated (kept the END of the prompt): 12

SELF-TEST (KV-cache decode vs full forward)
  A(300 tok): max|logit diff| = 0.0078 | same top-1: True | ref top-1 in cached top-3: True
  B(180 tok): max|logit diff| = 0.0078 | same top-1: True | ref top-1 in cached top-3: 

In [3]:
# ================================================================
# BUILD THE SUBMISSION FOLDER
#
# Reads the predictions written by infer_arch4.py, VALIDATES them against
# test.json (ids, order, count, exactly 128 words each, sanity checks), and
# writes a clean folder + zip:
#
#   submission/
#     predictions.json         [{"id": "1", "prediction": "..."}, ...]   (test order)
#     predictions_by_id.json   {"1": "...", "2": "...", ...}             (mirrors test.json's id -> item shape)
#     predictions.jsonl        one {"id", "prediction"} object per line
#     predictions.csv          id,prediction
#     predictions.txt          one prediction per line, test order
#     metadata.json            counts, word stats, sha256 of every file, model + decoding info
#     README.md                short model card (from-scratch statement, how it was produced)
#     code/                    training / inference scripts (if found on disk)
#     model/                   best.pt + tokenizer.json  (only if INCLUDE_WEIGHTS = True)
#   submission.zip
#
# Pure Python (no torch needed). Use the file whose format your competition
# asks for; delete the others or leave them, they are all consistent.
# ================================================================

import os, sys, json, csv, glob, shutil, hashlib, zipfile, time, re
from collections import Counter


def env(name, default):
    return os.environ.get(name, default)


# ----------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------
PRED_DIR = env("PRED_DIR", "/kaggle/working/inference_outputs")
SUB_DIR = env("SUB_DIR", "/kaggle/working/submission")
DATA_DIR = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1"
TEST_JSON_CANDIDATES = [os.path.join(DATA_DIR, "test.json"),
                        "/kaggle/input/datasets/achiver07/data-project/test.json"]
if env("TEST_JSON", ""):
    TEST_JSON_CANDIDATES = [env("TEST_JSON", "")]

TARGET_WORDS = 128
FORMATS = ["json_records", "json_by_id", "jsonl", "csv", "txt"]   # which prediction files to write
MAKE_ZIP = True

INCLUDE_CODE = True
CODE_FILES = ["train_arch4_t4_v4.py", "infer_arch4.py", "eval_generate_arch4.py", "make_submission.py"]
CODE_SEARCH_DIRS = ["/kaggle/working", os.getcwd()]

INCLUDE_WEIGHTS = False       # copies best.pt (~0.8 GB) + tokenizer.json into model/ (and into the zip)
WEIGHTS_SEARCH = ["/kaggle/input/**/best.pt", "/kaggle/working/**/best.pt"]

DECODING = {                  # what produced these predictions (edit if you used another config)
    "name": "mbr8_t0.7",
    "description": "8 sampled continuations per prompt (temperature 0.7, top-p 0.95, repetition "
                   "penalty 1.05 on generated tokens only, no repeated 4-grams); the candidate with "
                   "the highest average ROUGE-L agreement with the others is kept; truncated to "
                   "128 whitespace words; prompt tail (<=724 tokens) used as context.",
}

MODEL_FACTS = {
    "architecture": "decoder-only Transformer, 12 layers, d_model 1024, 16 query / 8 KV heads (GQA), "
                    "RoPE, SwiGLU (d_ff 3328), pre-LayerNorm, tied input/output embeddings; "
                    "sliding-window attention (256) with full attention every 3rd layer",
    "parameters": 201_443_328,
    "vocab_size": 40_000,
    "context_length": 1024,
    "training": "from scratch (random init), 10,000 steps x 16,384 tokens = 163.84M tokens, "
                "AdamW (lr 3e-4 cosine to 3e-5, 500 warmup steps, weight decay 0.1), fp16 mixed "
                "precision, one Tesla T4, 4.37 h",
    "validation_loss": 2.4760,
    "validation_perplexity": 11.89,
    "pretrained_components": "none (no pretrained weights, no pretrained libraries)",
}

# ----------------------------------------------------------------
# LOAD
# ----------------------------------------------------------------
ID_KEYS = ["id", "idx", "example_id", "uid", "index"]


def load_test_ids(path):
    raw = open(path, "r", encoding="utf-8").read()
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        data = [json.loads(l) for l in raw.splitlines() if l.strip()]
    if isinstance(data, dict):
        for k in ("data", "examples", "test", "instances", "items", "records"):
            if isinstance(data.get(k), (list, dict)) and len(data[k]) > 1:
                data = data[k]
                break
    if isinstance(data, dict):
        return list(data.keys())
    ids = []
    for i, ex in enumerate(data):
        ikey = next((k for k in ID_KEYS if isinstance(ex, dict) and k in ex), None)
        ids.append(ex[ikey] if ikey else i)
    return ids


def find_test_ids():
    for p in TEST_JSON_CANDIDATES:
        if os.path.exists(p):
            try:
                ids = load_test_ids(p)
                if ids:
                    print(f"test ids: {len(ids)} from {p}")
                    return ids
            except Exception as e:
                print(f"could not read {p}: {str(e)[:100]}")
    sys.exit("ERROR: could not read ids from test.json (checked: %s)" % TEST_JSON_CANDIDATES)


def load_predictions(test_ids):
    """Returns {str(id): prediction}."""
    rec = os.path.join(PRED_DIR, "predictions_records.json")
    jl = os.path.join(PRED_DIR, "predictions.jsonl")
    lst = os.path.join(PRED_DIR, "predictions_list.json")
    if os.path.exists(rec):
        print("reading", rec)
        return {str(r["id"]): r["prediction"] for r in json.load(open(rec, encoding="utf-8"))}
    if os.path.exists(jl):
        print("reading", jl)
        return {str(r["id"]): r["prediction"]
                for r in (json.loads(l) for l in open(jl, encoding="utf-8") if l.strip())}
    if os.path.exists(lst):
        print("reading", lst, "(ids taken from test.json order)")
        preds = json.load(open(lst, encoding="utf-8"))
        return {str(i): p for i, p in zip(test_ids, preds)}
    sys.exit(f"ERROR: no predictions found in {PRED_DIR}. Run infer_arch4.py with RUN_TEST = True first.")


# ----------------------------------------------------------------
# VALIDATE
# ----------------------------------------------------------------
_ctrl = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


def validate(test_ids, pred_map):
    errors, warnings = [], []
    sid = [str(i) for i in test_ids]
    if len(set(sid)) != len(sid):
        errors.append("test.json has duplicate ids")
    missing = [i for i in sid if i not in pred_map]
    extra = [i for i in pred_map if i not in set(sid)]
    if missing:
        errors.append(f"{len(missing)} test ids have no prediction (first: {missing[:5]})")
    if extra:
        warnings.append(f"{len(extra)} predictions have ids not in test.json (ignored; first: {extra[:5]})")

    wc, bad_type, empty, ctrl, repl, degenerate = [], [], [], [], [], []
    for i in sid:
        p = pred_map.get(i)
        if p is None:
            continue
        if not isinstance(p, str):
            bad_type.append(i)
            continue
        words = p.split()
        wc.append(len(words))
        if not words:
            empty.append(i)
        if _ctrl.search(p):
            ctrl.append(i)
        if "\ufffd" in p:
            repl.append(i)
        if len(words) >= 8:
            grams = [tuple(words[k:k + 4]) for k in range(len(words) - 3)]
            if len(set(grams)) / len(grams) < 0.6:
                degenerate.append(i)
    if bad_type:
        errors.append(f"{len(bad_type)} predictions are not strings (first ids: {bad_type[:5]})")
    if empty:
        errors.append(f"{len(empty)} predictions are empty (first ids: {empty[:5]})")
    off = sum(1 for c in wc if c != TARGET_WORDS)
    if off:
        warnings.append(f"{off} predictions do not have exactly {TARGET_WORDS} words "
                        f"(min {min(wc)}, max {max(wc)})")
    if ctrl:
        warnings.append(f"{len(ctrl)} predictions contain control characters (first: {ctrl[:5]})")
    if repl:
        warnings.append(f"{len(repl)} predictions contain the U+FFFD replacement character "
                        f"(a token split a multi-byte character; first: {repl[:5]})")
    if degenerate:
        warnings.append(f"{len(degenerate)} predictions look repetitive (<60% unique 4-grams; "
                        f"first: {degenerate[:5]})")
    dup = [p for p, c in Counter(pred_map[i] for i in sid if i in pred_map).items() if c > 1]
    if dup:
        warnings.append(f"{len(dup)} distinct prediction texts are used for more than one id")
    stats = {"examples": len(sid), "words_min": min(wc) if wc else 0,
             "words_avg": round(sum(wc) / len(wc), 2) if wc else 0, "words_max": max(wc) if wc else 0}
    return errors, warnings, stats


# ----------------------------------------------------------------
# WRITE
# ----------------------------------------------------------------
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def write_predictions(test_ids, pred_map):
    items = [(i, pred_map[str(i)]) for i in test_ids]
    written = []
    if "json_records" in FORMATS:
        p = os.path.join(SUB_DIR, "predictions.json")
        json.dump([{"id": i, "prediction": t} for i, t in items], open(p, "w", encoding="utf-8"),
                  ensure_ascii=False, indent=1)
        written.append(p)
    if "json_by_id" in FORMATS:
        p = os.path.join(SUB_DIR, "predictions_by_id.json")
        json.dump({str(i): t for i, t in items}, open(p, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        written.append(p)
    if "jsonl" in FORMATS:
        p = os.path.join(SUB_DIR, "predictions.jsonl")
        with open(p, "w", encoding="utf-8") as f:
            for i, t in items:
                f.write(json.dumps({"id": i, "prediction": t}, ensure_ascii=False) + "\n")
        written.append(p)
    if "csv" in FORMATS:
        p = os.path.join(SUB_DIR, "predictions.csv")
        with open(p, "w", encoding="utf-8", newline="") as f:
            w = csv.writer(f)
            w.writerow(["id", "prediction"])
            for i, t in items:
                w.writerow([i, t])
        written.append(p)
    if "txt" in FORMATS:
        p = os.path.join(SUB_DIR, "predictions.txt")
        with open(p, "w", encoding="utf-8") as f:
            for _, t in items:
                f.write(" ".join(t.split()) + "\n")
        written.append(p)
    return written


README = """# Phase 1 - Legal Language Modelling: submission

Predictions for {n} test prompts: one continuation of exactly {words} whitespace words per prompt.

## Model
{arch}

- Parameters: {params:,}
- Training: {training}
- Validation loss {vl} (perplexity {vp}) on the held-out validation split.
- Pretrained components: {pre}.

## How the predictions were produced
{dec}

## Files
- `predictions.json`: list of `{{"id", "prediction"}}` records in test order
- `predictions_by_id.json`: `{{id: prediction}}` mapping
- `predictions.jsonl`, `predictions.csv`, `predictions.txt`: the same predictions in other layouts
- `metadata.json`: counts, word statistics, SHA-256 of every file
- `code/`: training and inference scripts (if present)
- `model/`: weights + tokenizer (only if included)

Generated on {date}.
"""


def copy_code():
    copied = []
    os.makedirs(os.path.join(SUB_DIR, "code"), exist_ok=True)
    for name in CODE_FILES:
        for d in CODE_SEARCH_DIRS:
            src = os.path.join(d, name)
            if os.path.exists(src):
                shutil.copy2(src, os.path.join(SUB_DIR, "code", name))
                copied.append(name)
                break
    if not copied:
        os.rmdir(os.path.join(SUB_DIR, "code"))
        print("note: no code files found on disk (your notebook cells are not files). Save the scripts "
              "next to this one, or upload them into submission/code/ yourself.")
    else:
        missing = [n for n in CODE_FILES if n not in copied]
        print("code copied:", copied, "| not found:", missing)
    return copied


def copy_weights():
    hits = []
    for pat in WEIGHTS_SEARCH:
        hits += glob.glob(pat, recursive=True)
    if not hits:
        print("note: best.pt not found, skipping weights")
        return []
    os.makedirs(os.path.join(SUB_DIR, "model"), exist_ok=True)
    out = []
    shutil.copy2(hits[0], os.path.join(SUB_DIR, "model", "best.pt"))
    out.append("model/best.pt")
    tk = os.path.join(DATA_DIR, "tokenizer.json")
    if os.path.exists(tk):
        shutil.copy2(tk, os.path.join(SUB_DIR, "model", "tokenizer.json"))
        out.append("model/tokenizer.json")
    print("weights copied from", hits[0])
    return out


def tree(root):
    print(f"\n{root}/")
    for dp, dn, fn in sorted(os.walk(root)):
        rel = os.path.relpath(dp, root)
        depth = 0 if rel == "." else rel.count(os.sep) + 1
        if rel != ".":
            print("  " * depth + os.path.basename(dp) + "/")
        for f in sorted(fn):
            size = os.path.getsize(os.path.join(dp, f))
            print("  " * (depth + 1) + f"{f}  ({size / 1024:,.0f} KB)" if size < 5e7
                  else "  " * (depth + 1) + f"{f}  ({size / 1e6:,.0f} MB)")


def main():
    test_ids = find_test_ids()
    pred_map = load_predictions(test_ids)
    errors, warnings, stats = validate(test_ids, pred_map)

    print("\nVALIDATION")
    print(f"  examples: {stats['examples']} | words per prediction min/avg/max = "
          f"{stats['words_min']}/{stats['words_avg']}/{stats['words_max']}")
    for w in warnings:
        print("  WARNING:", w)
    for e in errors:
        print("  ERROR:  ", e)
    if errors:
        sys.exit("\nSubmission NOT built: fix the errors above (re-run infer_arch4.py if needed).")
    if not warnings:
        print("  all checks passed")

    if os.path.exists(SUB_DIR):
        shutil.rmtree(SUB_DIR)
    os.makedirs(SUB_DIR)

    written = write_predictions(test_ids, pred_map)
    extra_files = []
    if INCLUDE_CODE:
        extra_files += ["code/" + n for n in copy_code()]
    if INCLUDE_WEIGHTS:
        extra_files += copy_weights()

    with open(os.path.join(SUB_DIR, "README.md"), "w", encoding="utf-8") as f:
        f.write(README.format(n=stats["examples"], words=TARGET_WORDS, arch=MODEL_FACTS["architecture"],
                              params=MODEL_FACTS["parameters"], training=MODEL_FACTS["training"],
                              vl=MODEL_FACTS["validation_loss"], vp=MODEL_FACTS["validation_perplexity"],
                              pre=MODEL_FACTS["pretrained_components"], dec=DECODING["description"],
                              date=time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime())))

    files = [os.path.basename(p) for p in written] + extra_files + ["README.md"]
    meta = {
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "validation": {"stats": stats, "warnings": warnings},
        "model": MODEL_FACTS, "decoding": DECODING,
        "sha256": {n: sha256(os.path.join(SUB_DIR, n)) for n in files},
    }
    json.dump(meta, open(os.path.join(SUB_DIR, "metadata.json"), "w", encoding="utf-8"), indent=2)

    if MAKE_ZIP:
        zpath = SUB_DIR.rstrip("/") + ".zip"
        if os.path.exists(zpath):
            os.remove(zpath)
        with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
            for dp, _, fn in os.walk(SUB_DIR):
                for f in fn:
                    full = os.path.join(dp, f)
                    z.write(full, os.path.relpath(full, SUB_DIR))
        print(f"\nzip: {zpath} ({os.path.getsize(zpath) / 1e6:.1f} MB)")

    tree(SUB_DIR)
    print("\nDone. Download the folder/zip from /kaggle/working (or Save Version -> Output).")


main()

test ids: 1000 from /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
reading /kaggle/working/inference_outputs/predictions_records.json

VALIDATION
  examples: 1000 | words per prediction min/avg/max = 128/128.0/128
note: no code files found on disk (your notebook cells are not files). Save the scripts next to this one, or upload them into submission/code/ yourself.

zip: /kaggle/working/submission.zip (1.1 MB)

/kaggle/working/submission/
  README.md  (2 KB)
  metadata.json  (2 KB)
  predictions.csv  (701 KB)
  predictions.json  (733 KB)
  predictions.jsonl  (724 KB)
  predictions.txt  (693 KB)
  predictions_by_id.json  (704 KB)

Done. Download the folder/zip from /kaggle/working (or Save Version -> Output).
